# 02 · Factorization & Scaling Experiments

The experiment stage: the pre-registered hypotheses, the run matrix and its shared cell,
the launches (RUN LATER), the registry-driven analysis — leave-one-out Δ with the seed
band, the interaction gap, the data-scaling curve with its log₂ fit and secant slopes,
the one val→test check — and the **pre-written interpretation branches** we commit to
now, before any result exists. Verdicts get filled in when the runs land; the reasoning
is frozen here so the outcome cannot rewrite the question.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §2 (hypotheses), §3 (design &
  run accounting), §6 (protocol), §13 (interpretation matrix).
- **Registry:** `../results/registry.csv` (columns include `aug_remix, aug_gain,
  aug_flip, n_songs`); eval CSVs: `../results/*_per_track.csv`.
- **Shared cell:** the FULL-86 anchor is Direction 01's `l1mag` sweep cell, **reused not
  retrained** — the config hashes are asserted equal by
  `tests/test_config_schema.py` (gate G0), so no FULL-86 job appears in this direction's
  launch list.
- **Discipline:** the 50 test tracks are read **exactly once** (§6, gate G3), on the
  FULL-86 checkpoints only, after the validation analysis below is frozen.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · Pre-registration recap (verbatim from MASTER_PLAN §2)

Let val-SI-SDR(c) = mean full-track vocals SI-SDR over the 14 validation tracks for
config c (final-vs-best checkpoint both logged; decisions use **best**, as in D01).
Let **σ_seed** = pooled between-seed standard deviation from the two 3-seed cells
(FULL-86 and n21), pooled as $\sigma_{\text{seed}}=\sqrt{(s^2_{\text{FULL}}+s^2_{\text{n21}})/2}$.

> **H-02a (remix dominance).** Define per-transform contributions
> $\Delta_t = \overline{\text{val}}(\text{FULL}) - \text{val}(\text{FULL} \setminus t)$
> for $t \in$ {remix, gain, flip}.
> - **Supported** iff $\Delta_{\text{remix}} > \max(\Delta_{\text{gain}}, \Delta_{\text{flip}}) + \sigma_{\text{seed}}$
>   **and** $\Delta_{\text{remix}} > \sigma_{\text{seed}}$.
> - **Refuted** iff $\Delta_{\text{remix}} \le \max(\Delta_{\text{gain}}, \Delta_{\text{flip}})$
>   (remix is not the top lever) **or** $\Delta_{\text{remix}} \le \sigma_{\text{seed}} \wedge \Delta_{\text{total}} \le \sigma_{\text{seed}}$
>   (augmentation as a whole is worthless at this budget — see descriptive quantities).
> - **Mixed** otherwise (e.g. remix on top but within the band).
> This is a **single pre-registered contrast** (remix vs the best of the others), so no
> multiple-comparison correction is applied to the primary verdict; the full Δ table is
> descriptive.

> **H-02b (unsaturated scaling).** With the full recipe and nested subsets
> $N \in \{21, 43, 64, 86\}$:
> - **Supported** iff $\overline{\text{val}}(86) - \text{val}(43) > \sigma_{\text{seed}}$
>   (the last measured **doubling** of data still buys more than run-to-run noise) and
>   the four-point sequence is non-decreasing within $\pm\sigma_{\text{seed}}$ tolerance.
> - **Refuted (plateau)** iff $\overline{\text{val}}(86) - \text{val}(43) \le \sigma_{\text{seed}}/2$
>   — at this capacity/budget, MUSDB's data volume is no longer the binding constraint.
> - **Mixed** otherwise (borderline gain, or a non-monotone middle worse than the
>   tolerance — see §12 subset-sensitivity risk).

**Descriptive quantities (no hypotheses, reported with CIs):**
$\Delta_{\text{total}} = \overline{\text{val}}(\text{FULL}) - \text{val}(\text{NONE})$
(total recipe value); the interaction gap
$\Delta_{\text{total}} - \sum_t \Delta_t$ (transform synergy/redundancy); the log-fit
slope $b$ in $\widehat{\text{SISDR}}(N) = a + b \log_2 N$ (dB per doubling of songs,
least-squares on the four means, parametric-bootstrap CI using σ_seed as per-point
noise); the same-loss D01 FULL-86 arm (`l1mag_seed*_reduced`) cross-check — Direction
01's `l1mag` sweep cell **is** this direction's FULL-86 cell (§3.3).

**Scope note (pre-registered deviation from the RESEARCH_DIRECTIONS #2 sketch):** the
menu sketch mentioned pitch/tempo among candidate transforms. Our frozen recipe
(D01 §4.5, PLAN Phase 2) deliberately excludes pitch/tempo (heavyweight, separate code
path even in Demucs) and channel-swap (inapplicable to the mono pipeline — documented,
not silently dropped). The factorization covers exactly the three transforms the
project actually trains with.

## 2 · Run matrix & the shared cell (MASTER_PLAN §3.1–3.3)

**Factorization block (leave-one-out, all on the full 86-song split):**

| Config | remix | gain | flip | Seeds | New runs |
|---|---|---|---|---|---|
| `full` (≡ n86) | ✓ | ✓ | ✓ | {0, 1, 2} | **0** (≡ D01 `l1mag` cell, reused) |
| `no_remix` | ✗ | ✓ | ✓ | {0} | 1 |
| `no_gain` | ✓ | ✗ | ✓ | {0} | 1 |
| `no_flip` | ✓ | ✓ | ✗ | {0} | 1 |
| `none` | ✗ | ✗ | ✗ | {0} | 1 |

**Scaling block (full recipe, nested subsets, identical 16 k-step budget):**

| Config | Songs | Seeds | New runs |
|---|---|---|---|
| `n21` | 21 | {0, 1, 2} | 3 |
| `n43` | 43 | {0} | 1 |
| `n64` | 64 | {0} | 1 |
| `n86` ≡ `full` | 86 | {0, 1, 2} | **0** (shared) |

**Accounting (§3.3):** new GPU runs = 4 (LOO) + 5 (scaling) = **9**; the two 3-seed
anchor cells (FULL-86, n21) pin σ_seed. Worst case (if the D01 cell is unavailable or
its hash mismatches) it re-runs as +3 → 12. Budget ceiling ≈ **13–25 T4-GPU-hours**
(§7). The `full`/`n86` cell is intentionally absent from the launch list below.

### 2.1 Pre-flight: dry-run the switchboard

> ⚠️ **RUN THIS LATER** — configuration check · _CPU, minutes · no GPU, no data_

Before any GPU spend, confirm every config builds the pipeline it claims to. The dry-run
instantiates each config's `AugmentPipeline` and prints the remix/gain/flip switches, the
subset size, and the config hash — abort on any mismatch with the run matrix above
(gate G1). The shared FULL-86 hash it prints must equal Direction 01's `l1mag_seed0_reduced`.

In [ ]:
# ⚠️ RUN THIS LATER (CPU, minutes) — prints the switchboard + subset size per config.
# !python scripts/run_sweep.py --direction 02 --stage reduced --dry-run
print("Dry-run prints the 9-config switchboard table (RUN LATER; CPU, no GPU).")

### 2.2 Launch the 9 runs

> ⚠️ **RUN THIS LATER** — 4 LOO (5–7 h) + 5 scaling (7–9 h) · _≈ 13–25 T4-h total · resumable_

Resumable: a completed config is skipped and an interrupted one resumes from its last
checkpoint, so a Colab disconnect costs minutes (§7 run book).

In [ ]:
# ⚠️ RUN THIS LATER (GPU). Iterates the 9 configs; skips completed runs.
# !python scripts/run_sweep.py --direction 02 --stage reduced
print("The 9 reduced-budget runs are RUN LATER (GPU).")

## 3 · Registry-driven analysis — leave-one-out (H-02a)

These cells read `../results/registry.csv` and call `singnet.analysis.scaling` — no
analysis logic lives in the notebook. They fill the moment the registry has rows; until
then they print a placeholder. The FULL-86 rows are the shared D01 `l1mag` cell (matched
by `aug_remix=aug_gain=aug_flip=True, n_songs=86`).

In [ ]:
import pandas as pd, numpy as np
from singnet.train import read_registry
from singnet.analysis import loo_table, pooled_seed_sigma

reg = read_registry('02-augmentation-data-scaling/results/registry.csv')
if len(reg):
    table = loo_table(reg)                         # Δ_t, Δ_total, interaction gap
    full = reg[(reg.aug_remix.astype(str).isin(['True','1','TRUE'])) &
               (reg.aug_gain.astype(str).isin(['True','1','TRUE'])) &
               (reg.aug_flip.astype(str).isin(['True','1','TRUE'])) & (reg.n_songs == 86)]
    n21 = reg[reg.n_songs == 21]
    sigma_seed = pooled_seed_sigma(full['best_val_sisdr'].to_numpy(), n21['best_val_sisdr'].to_numpy())
    print(f"pooled σ_seed = {sigma_seed:.3f} dB")
    display(table)
else:
    print('no runs yet — this table fills once run_sweep.py has populated the registry.')

### 3.1 LOO Δ bar chart with the σ_seed band

*Figure to render:* one bar per transform = its leave-one-out Δ (dB lost when removed
from the full recipe), with a shaded ±σ_seed band. **How to read it:** a bar clearing the
band is a resolved contribution; a bar inside it is within seed noise. Pre-registered
expectations: **remix** clears the band (H-02a), **flip** sits at ≈ 0 inside it (the
placebo — a flip bar *outside* the band is a red flag, §13). The interaction gap
(Δ_total − ΣΔ_t) is annotated: negative = redundancy, positive = synergy (THEORY §5.3).

In [ ]:
# Fills once the registry has the LOO rows. Draws the σ_seed band; flags the flip placebo.
import matplotlib.pyplot as plt
if len(reg):
    deltas = table.set_index('contrast').loc[['no_remix', 'no_gain', 'no_flip'], 'delta']
    ax = deltas.rename({'no_remix': 'remix', 'no_gain': 'gain', 'no_flip': 'flip'}).plot.bar()
    ax.set_ylabel('leave-one-out Δ  (dB val SI-SDR)')
    ax.set_title('Per-transform value with the ±σ_seed band')
    ax.axhspan(-sigma_seed, sigma_seed, alpha=0.15, label='±σ_seed')
    ax.axhline(0, lw=0.8); ax.legend()
    gap = float(table.set_index('contrast').loc['interaction_gap', 'delta'])
    ax.annotate(f'interaction gap = {gap:+.2f} dB', xy=(0.02, 0.92), xycoords='axes fraction')
else:
    print('no runs yet.')

## 4 · The data-scaling curve (H-02b)

*Figure to render:* mean val SI-SDR versus song-count on a log₂ x-axis, with the 3-seed
endpoint noise bands at n21 and n86, the least-squares log₂ fit drawn **only over
[21, 86]**, and the per-doubling secant slopes annotated. **How to read it:** the sign
and size of the slope at n86 is the finding. H-02b is *supported* if the last doubling
(43→86) buys more than σ_seed and the sequence is non-decreasing; *refuted (plateau)* if
that last gain is ≤ σ_seed/2. The secant slopes answer "is it bending?" without leaning
on the two-parameter form; any dashed extension past 86 is captioned
"descriptive extrapolation — no data beyond 86 songs" (§6).

In [ ]:
# Build (N, mean val) from the scaling rows and fit; all logic in singnet.analysis.scaling.
from singnet.analysis import fit_log2, secant_slopes
if len(reg):
    full_recipe = reg[reg.aug_remix.astype(str).isin(['True','1','TRUE']) &
                      reg.aug_gain.astype(str).isin(['True','1','TRUE']) &
                      reg.aug_flip.astype(str).isin(['True','1','TRUE'])]
    curve = full_recipe.groupby('n_songs')['best_val_sisdr'].mean().sort_index()
    ns, scores = curve.index.to_numpy(), curve.to_numpy()
    if len(ns) >= 2:
        fit = fit_log2(ns, scores, sigma=sigma_seed)      # parametric-bootstrap CI on b
        secants = secant_slopes(ns, scores)               # dB per doubling
        print(f"log2 fit: SISDR ≈ {fit.a:.2f} + {fit.b:.2f}·log2(N)  "
              f"[95% CI on b: {fit.b_ci[0]:.2f}, {fit.b_ci[1]:.2f}]")
        display(secants)
        # fig: curve with endpoint σ_seed bands + fit line over [21, 86] + secant labels
else:
    print('no scaling runs yet — curve fills from the registry.')

## 5 · The single test pass (FULL-86 → 50 test tracks)

> ⚠️ **RUN THIS LATER** — score the 3 FULL-86 checkpoints on 50 test tracks · _GPU minutes / CPU-ok_

**Gate G3:** run only after the validation analysis above is frozen and committed, and
only on the FULL-86 checkpoints (seeds 0–2) — the LOO and subset arms are never tested
(their questions are validation-scoped by design). This anchors the scaling curve's
endpoint on truly held-out data and measures the val→test generalization gap of the
headline cell.

In [ ]:
# ⚠️ RUN THIS LATER — the ONE test pass for this direction (§6). FULL-86 checkpoints only.
# !python scripts/evaluate.py --checkpoints <ckpt_seed0> <ckpt_seed1> <ckpt_seed2> \
#     --split test --shard-root $SHARD_ROOT --out 02-augmentation-data-scaling/results/
print("Single test pass on the FULL-86 checkpoints is RUN LATER (GPU minutes).")

### 5.1 val→test generalization of the FULL-86 cell

*Fills once `../results/test_per_track.csv` exists.* Reports the mean test SI-SDR of the
FULL-86 cell against its validation number — a single honest generalization gap for the
headline cell, with a per-track bootstrap CI (the endpoint of the scaling curve on
held-out data).

In [ ]:
from pathlib import Path
csv = Path('02-augmentation-data-scaling/results/test_per_track.csv')
if csv.exists():
    test = pd.read_csv(csv)
    print('mean test vocals SI-SDR (FULL-86):', round(float(test['sisdr_vocals'].mean()), 3), 'dB')
    # compare to the FULL-86 val number from the registry -> the generalization gap
else:
    print('test CSV not present yet — the single test pass is RUN LATER (§5).')

## 6 · Contingency — loss sensitivity (MASTER_PLAN §3.4)

> ⚠️ **RUN THIS LATER** — only if Direction 01 replaces the default loss · _2 × ~1.5 h_

If Direction 01's verdict flips the project default to `sisdr`, re-run `full` and `none`
under `sisdr` (seed 0) to check the factorization's **sign structure** is loss-stable —
reported as a sensitivity appendix. This does **not** alter the H-02a/b verdicts, which
are defined under `l1mag`.

In [ ]:
# ⚠️ RUN THIS LATER (GPU) — budget-gated; only if D01 overturned the default loss.
# !python scripts/run_sweep.py --direction 02 --stage contingency
print("Contingency (sisdr full vs none) is RUN LATER, only if D01 flipped the loss.")

## 7 · Interpretation — pre-written branches (select one when results exist)

_These are the pre-registered readings of MASTER_PLAN §13, written now, **before** the
runs, so the outcome cannot rewrite the question. When the registry is complete, keep the
one branch the numbers select and move it (adapted) into `paper/PAPER.md`._

### H-02a branches (remix dominance)
- **[branch: H-02a SUPPORTED]** — Δ_remix > max(Δ_gain, Δ_flip) + σ_seed and Δ_remix >
  σ_seed. *Reading:* remix is the small-data lever; the folklore is confirmed **with a
  measured effect size**. *Consequence:* remix stays mandatory everywhere, and Direction
  10's unlabeled-pool sizing gains a measured anchor.
- **[branch: H-02a REFUTED — remix ≤ others]** — remix is not the top lever. *Reading:*
  the mixture-manifold story is wrong at this scale; gain/flip-style perturbations
  suffice. *Consequence:* re-examine remix's role before D10; flag for replication at
  FULL budget.
- **[branch: H-02a REFUTED — nothing matters]** — Δ_remix ≤ σ_seed and Δ_total ≤ σ_seed.
  *Reading:* at 16 k steps the model is budget-bound, not data-bound. *Consequence:*
  rerun `full` vs `none` once at FULL budget before accepting (pre-registered +2 runs).
- **[branch: H-02a MIXED]** — remix on top but within the band. *Reading:* directionally
  consistent, not resolved. *Consequence:* report the Δ table descriptively; the G2
  escalation (seeds on `no_remix`) may resolve it.

### H-02b branches (unsaturated scaling)
- **[branch: H-02b SUPPORTED]** — val(86) − val(43) > σ_seed, sequence non-decreasing.
  *Reading:* MUSDB-only training is data-starved; the slope b quantifies the price of
  data. *Consequence:* strengthens D10 (distillation adds effective data); motivates
  extra-data lines in any future work.
- **[branch: H-02b REFUTED — plateau]** — val(86) − val(43) ≤ σ_seed/2. *Reading:* 86
  songs saturate this capacity; the constraint is the model, not the data. *Consequence:*
  strengthens D03 (capacity/architecture is the lever); D10's premise weakens (state so).
- **[branch: H-02b MIXED — non-monotone]** — a borderline gain or a middle point worse
  than tolerance. *Reading:* subset-draw sensitivity dominates. *Consequence:* report the
  per-subset balance; downgrade the curve to descriptive ("one nested draw").

### Flip-control branch (the design's placebo)
- **[branch: FLIP CONTROL FIRES]** — Δ_flip > σ_seed. *Reading:* a design fault — sign
  flip cannot help a magnitude model (THEORY §3.1), so this points at an RNG-stream leak,
  a normalization bug, or an unintended phase-sensitive term. *Consequence:* **halt
  interpretation of every other Δ**, debug, and record the cause in `results/DEVIATIONS.md`
  before trusting the LOO table or the curve.

## 8 · Conclusions & what this changes for directions 03–10

- The **data-vs-capacity** reading feeds **Direction 03** (mini band-split): a rising
  curve at 86 songs says "add data"; a plateau says "add capacity/architecture" — the
  scaling verdict tells D03 which lever it is pulling.
- The **remix value** feeds **Direction 10** (Demucs distillation): if remix dominates,
  the unlabeled/pseudo-label pool has a measured worth per added song, sizing D10's pool;
  if it does not, D10's "more effective data" premise weakens and we say so.
- The **σ_seed band** established here is the project's noise ruler for every later sweep
  figure (house style: no difference is claimed inside the band).
- Whatever the outcome — remix dominates, augmentation barely matters, the curve rises or
  plateaus — it is a pre-registered, reportable finding. *The shape is the result,
  whichever way it bends.*